# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanoleo/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import files
uploaded = files.upload()  # select capstone_features.csv

import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv('capstone_features.csv')
feats = ['imp_prev30','visible_queries','rare_share','anon_share','top_query_share','pos_volatility_60d']
model_data = df.dropna(subset=feats).copy()
X, y = model_data[feats], model_data['is_declining']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=model_data['client_hash_id']))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
print(f'{len(model_data):,} rows ready, model trained')

Saving capstone_features.csv to capstone_features.csv
102,202 rows ready, model trained


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [2]:
test_data = model_data.loc[X_te.index].copy()
test_data['decline_score'] = model.predict_proba(X_te)[:, 1]

q75_vol = test_data['pos_volatility_60d'].quantile(0.75)
q75_rare = test_data['rare_share'].quantile(0.75)
q25_imp = test_data['imp_prev30'].quantile(0.25)
q75_top = test_data['top_query_share'].quantile(0.75)

def reason_code(row):
    reasons = []
    if row['pos_volatility_60d'] > q75_vol: reasons.append('unstable ranking position')
    if row['rare_share'] > q75_rare: reasons.append('relies on rare/long-tail queries')
    if row['imp_prev30'] < q25_imp: reasons.append('already low impressions')
    if row['top_query_share'] > q75_top: reasons.append('over-reliant on one query')
    return '; '.join(reasons) if reasons else 'moderate signals, no single driver'

test_data['reason_codes'] = test_data.apply(reason_code, axis=1)
ranked = test_data.sort_values('decline_score', ascending=False)
print(f'Total scored: {len(ranked):,} | Top decile threshold: {ranked["decline_score"].quantile(0.9):.3f}')
print(ranked[['client_hash_id','decline_score','reason_codes']].head(10).to_string(index=False))

Total scored: 51,701 | Top decile threshold: 0.835
         client_hash_id  decline_score                                                                                                    reason_codes
client_fef1a8f436438636            1.0                                                            unstable ranking position; over-reliant on one query
client_73cda7b4e4f265ea            1.0                                                            unstable ranking position; over-reliant on one query
client_fef1a8f436438636            1.0                                   unstable ranking position; already low impressions; over-reliant on one query
client_fef1a8f436438636            1.0                                                                                       over-reliant on one query
client_e5c2aa26a8598242            1.0                                                                              moderate signals, no single driver
client_73cda7b4e4f265ea            1.0 unst

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [3]:
print("Intended use: prioritize WHICH pages a human reviews first.")
print("Not intended: automatically deciding a page's fate, or treating the")
print("score as a probability guarantee. ROC-AUC 0.705 means meaningfully")
print("better than chance, not near-certain.")

Intended use: prioritize WHICH pages a human reviews first.
Not intended: automatically deciding a page's fate, or treating the
score as a probability guarantee. ROC-AUC 0.705 means meaningfully
better than chance, not near-certain.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [4]:
print("No-go: never auto-remove, auto-unpublish, or auto-deprioritize a page")
print("based on this score alone. Always route to human review first.")
print("Every flagged page needs a person to read the reason code and confirm")
print("it makes sense before any action is taken.")

No-go: never auto-remove, auto-unpublish, or auto-deprioritize a page
based on this score alone. Always route to human review first.
Every flagged page needs a person to read the reason code and confirm
it makes sense before any action is taken.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [5]:
print("Retrain trigger 1: a new month of warehouse data becomes available.")
print("Retrain trigger 2: held-out AUC on a fresh evaluation drifts below ~0.65,")
print("suggesting the patterns the model learned no longer hold.")

Retrain trigger 1: a new month of warehouse data becomes available.
Retrain trigger 2: held-out AUC on a fresh evaluation drifts below ~0.65,
suggesting the patterns the model learned no longer hold.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
print("Artifacts reused in the deployed paper: chart_feature_importance.png,")
print("chart_roc_curve.png, and the Results table (baseline vs. model AUC),")
print("all built and saved in capstone.ipynb.")

Artifacts reused in the deployed paper: chart_feature_importance.png,
chart_roc_curve.png, and the Results table (baseline vs. model AUC),
all built and saved in capstone.ipynb.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.